# 실습04 · 클래스 불균형 대응 (CrossEntropy weight)

> 원본 스크립트: `02_응용/04_weight.py` · PPT 47쪽 · TODO A04

## 학습 목표
- train 개수의 역수로 클래스 가중치를 만들고 평균 1로 맞춘다.
- 같은 head 모델에서 `none`과 `weight` 두 조건만 바꾸어 비교한다.
- Accuracy와 macro F1, minority(triangle) Recall이 서로 다른 정보를 준다는 것을 확인한다.

| 클래스 | train 개수 | 1/개수 | 평균 1 정규화 가중치 |
|---|---|---|---|
| circle | 120 | 0.00833 | 0.3 |
| square | 40 | 0.025 | 0.9 |
| triangle | 20 | 0.05 | 1.8 |

In [1]:
# ============================================================
# 0. 노트북 실행 환경 준비
# ============================================================

# 파일·폴더 경로를 객체로 다루는 Path
from pathlib import Path
# 결과 목록을 JSON 파일로 저장
import json
# 수치 배열 계산과 source 가중치(npz) 읽기
import numpy as np
# 결과 표(DataFrame) 생성·CSV 저장
import pandas as pd
# PyTorch: Tensor 연산, 자동 미분, 난수 seed
import torch
# 신경망 층(Conv2d, Linear 등)과 손실 함수
from torch import nn
# data/분할/클래스 폴더 구조를 데이터셋으로 읽는 ImageFolder
from torchvision.datasets import ImageFolder
# 이미지 변환(Resize, ToTensor) 묶음
from torchvision import transforms as tr
# 분류 평가 지표: 정확도, F1, 클래스별 보고서, 혼동행렬
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
# Matplotlib: 노트북 안에 이미지·그래프 표시
import matplotlib.pyplot as plt

# CPU 스레드 1개, seed 14 고정 -> .py 스크립트와 같은 재현 조건
torch.set_num_threads(1)
torch.manual_seed(14)

# 노트북은 자기 폴더(02_응용/)를 작업 경로로 실행된다
# - DATA : 원본 입력 (읽기만 하고 덮어쓰지 않음)
# - OUT  : 노트북이 만드는 결과물 전용 폴더 (스크립트의 outputs/와 분리)
DATA = Path("data")
OUT = Path("outputs")
OUT.mkdir(exist_ok=True)

# [0,1] RGB 32×32 -> 소형 source Backbone의 학습 입력 조건
# 주의: ImageNet Normalize는 넣지 않음
transform = tr.Compose([tr.Resize((32, 32)), tr.ToTensor()])
print("torch", torch.__version__, "/ 결과 폴더:", OUT)

torch 2.14.0+cu130 / 결과 폴더: outputs


In [2]:
# ============================================================
# 1. train / valid 이미지를 Tensor 묶음으로 준비
# ============================================================

# train : 학습에만 사용, valid : epoch·후보 비교에만 사용
# 주의: test 폴더는 읽지 않음
train = ImageFolder(DATA / "train", transform=transform)
valid = ImageFolder(DATA / "valid", transform=transform)

# torch.stack : 이미지 N장을 하나의 배치 Tensor로 쌓음
X_train = torch.stack([train[i][0] for i in range(len(train))])
y_train = torch.tensor(train.targets, dtype=torch.long)
X_valid = torch.stack([valid[i][0] for i in range(len(valid))])
y_valid = torch.tensor(valid.targets, dtype=torch.long)

print("classes:", train.class_to_idx)
print("X_train:", tuple(X_train.shape), "/ y_train 개수:", torch.bincount(y_train).tolist())
print("X_valid:", tuple(X_valid.shape), "/ y_valid 개수:", torch.bincount(y_valid).tolist())
print("값 범위:", round(float(X_train.min()), 3), "~", round(float(X_train.max()), 3))

classes: {'circle': 0, 'square': 1, 'triangle': 2}
X_train: (180, 3, 32, 32) / y_train 개수: [120, 40, 20]
X_valid: (90, 3, 32, 32) / y_valid 개수: [30, 30, 30]
값 범위: 0.024 ~ 0.937


In [3]:
# ============================================================
# 2. 모델 생성 함수: TinyShapeCNN Backbone + 새 Head
# ============================================================

def build_model():
    """source Backbone을 불러온 TinyShapeCNN을 만든다.

    - 0~8번 층: source_backbone.npz 값으로 덮어씀 (사전학습 Backbone)
    - 9번 층 Linear(256, 3): 현재 seed로 새로 초기화된 Head
    """
    model = nn.Sequential(
        nn.Conv2d(3, 8, 3, padding=1, bias=False), nn.BatchNorm2d(8), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(8, 16, 3, padding=1, bias=False), nn.BatchNorm2d(16), nn.ReLU(),
        nn.AdaptiveAvgPool2d((4, 4)), nn.Flatten(), nn.Linear(256, 3),
    )
    # allow_pickle=False : 수치 배열만 읽음 (파이썬 객체 복원 금지)
    with np.load(DATA / "source_backbone.npz", allow_pickle=False) as saved:
        state = model.state_dict()
        state.update({name: torch.from_numpy(saved[name].copy()) for name in saved.files})
    model.load_state_dict(state)
    return model


torch.manual_seed(14)
m = build_model()
x = X_valid[:1]
# 층을 하나씩 통과시키며 shape 변화를 확인
for i, layer in enumerate(m):
    x = layer(x)
    print(f"[{i}] {layer.__class__.__name__:<18} -> {tuple(x.shape)}")

[0] Conv2d             -> (1, 8, 32, 32)
[1] BatchNorm2d        -> (1, 8, 32, 32)
[2] ReLU               -> (1, 8, 32, 32)
[3] MaxPool2d          -> (1, 8, 16, 16)
[4] Conv2d             -> (1, 16, 16, 16)
[5] BatchNorm2d        -> (1, 16, 16, 16)
[6] ReLU               -> (1, 16, 16, 16)
[7] AdaptiveAvgPool2d  -> (1, 16, 4, 4)
[8] Flatten            -> (1, 256)
[9] Linear             -> (1, 3)


In [4]:
# ============================================================
# 3. 후보 하나를 학습·평가하는 함수 (스크립트 반복문 본문과 같은 로직)
# ============================================================

import time

def run_candidate(strategy, balance, keep=False):
    """(strategy, balance) 후보 하나를 8 epoch 학습하고 valid 지표를 반환한다.

    - strategy="head"    : 새 Head(model[9])만 학습
    - strategy="partial" : 마지막 Conv(model[4]) + Head 학습
    - balance="none"     : 기본 CrossEntropy
    - balance="weight"   : train 개수 역수 기반 클래스 가중치 CrossEntropy
    - keep=True          : 모델·확률·예측도 함께 반환
    """
    torch.manual_seed(14)
    model = build_model()

    # --------------------------------------------------------
    # 학습 범위: requires_grad로 갱신 대상 지정
    # --------------------------------------------------------
    for p in model.parameters():
        p.requires_grad = False
    for p in model[9].parameters():
        p.requires_grad = True
    if strategy == "partial":
        for p in model[4].parameters():
            p.requires_grad = True
    # BN running 통계 고정 (gradient는 끄지 않음)
    model.eval()

    # lr=0.003 : 학습률, weight_decay=0.0001 : 가중치 감쇠 규제
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=0.003, weight_decay=0.0001)
    counts = torch.bincount(y_train, minlength=3).float()
    inverse = 1.0 / counts
    class_weights = inverse / inverse.mean()
    criterion = nn.CrossEntropyLoss(weight=class_weights if balance == "weight" else None)

    # --------------------------------------------------------
    # 8 epoch 학습, valid 손실 최소 epoch 보관
    # --------------------------------------------------------
    history, best_loss = [], float("inf")
    started = time.perf_counter()
    for epoch in range(1, 9):
        optimizer.zero_grad()
        loss = criterion(model(X_train), y_train)
        loss.backward()
        optimizer.step()
        with torch.no_grad():
            vl = nn.CrossEntropyLoss()(model(X_valid), y_valid).item()
        history.append({"strategy": strategy, "balance": balance, "epoch": epoch, "valid_loss": vl})
        if vl < best_loss:
            best_loss, best_epoch = vl, epoch
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    elapsed = time.perf_counter() - started

    # --------------------------------------------------------
    # best epoch 복원 후 valid 평가
    # --------------------------------------------------------
    model.load_state_dict(best_state)
    with torch.no_grad():
        prob = model(X_valid).softmax(1)
        pred = prob.argmax(1)
    report = classification_report(y_valid, pred, labels=[0, 1, 2], output_dict=True, zero_division=0)
    row = {
        "strategy": strategy,
        "balance": balance,
        "params": sum(p.numel() for p in model.parameters() if p.requires_grad),
        "best_epoch": best_epoch,
        "accuracy": accuracy_score(y_valid, pred),
        "macro_f1": f1_score(y_valid, pred, labels=[0, 1, 2], average="macro", zero_division=0),
        "triangle_recall": report["2"]["recall"],
        "seconds": elapsed,
    }
    if keep:
        return row, history, class_weights, model, prob, pred
    return row, history, class_weights

## 4. 가중치가 손실에 주는 영향

In [5]:
# ============================================================
# 4. 클래스 가중치 계산과 가중 평균 손실 확인
# ============================================================

counts = torch.bincount(y_train, minlength=3).float()
inverse = 1.0 / counts
class_weights = inverse / inverse.mean()
print("counts:", counts.tolist())
print("class_weights:", class_weights.tolist())

# 클래스별 총 가중치 = 개수 × 가중치 -> 세 클래스가 같은 총 비중(36)을 갖게 됨
print("개수 × 가중치:", (counts * class_weights).tolist())

# 주의: weight가 있으면 CrossEntropy는 sum(w_i * loss_i) / sum(w_i)로 평균함

counts: [120.0, 40.0, 20.0]
class_weights: [0.30000001192092896, 0.9000000357627869, 1.8000000715255737]
개수 × 가중치: [36.0, 36.0, 36.0]


## 5. TODO A04 · none vs weight

In [6]:
# ============================================================
# 5. TODO A04: 같은 head 모델에서 none, weight 비교
# ============================================================

experiments = [("head", "none"), ("head", "weight")]
rows, results = [], {}
for strategy, balance in experiments:
    row, history, class_weights, model, prob, pred = run_candidate(strategy, balance, keep=True)
    rows.append(row)
    results[balance] = pred

print("train class_weights:", class_weights.tolist())
table = pd.DataFrame(rows)
table.to_csv(OUT / "weight_comparison.csv", index=False)
print(table.drop(columns="seconds").round(4).to_string(index=False))

train class_weights: [0.30000001192092896, 0.9000000357627869, 1.8000000715255737]
strategy balance  params  best_epoch  accuracy  macro_f1  triangle_recall
    head    none     771           8    0.8111    0.8038              0.5
    head  weight     771           8    1.0000    1.0000              1.0


In [7]:
# 두 조건의 valid 혼동행렬 (행=실제, 열=예측)
names = valid.classes
for balance, pred in results.items():
    cm = pd.DataFrame(confusion_matrix(y_valid, pred, labels=[0, 1, 2]), index=names, columns=names)
    print(f"--- head/{balance} ---")
    print(cm)
    print(classification_report(y_valid, pred, labels=[0, 1, 2], target_names=names, zero_division=0, digits=4))

--- head/none ---
          circle  square  triangle
circle        30       0         0
square         2      28         0
triangle      15       0        15
              precision    recall  f1-score   support

      circle     0.6383    1.0000    0.7792        30
      square     1.0000    0.9333    0.9655        30
    triangle     1.0000    0.5000    0.6667        30

    accuracy                         0.8111        90
   macro avg     0.8794    0.8111    0.8038        90
weighted avg     0.8794    0.8111    0.8038        90

--- head/weight ---
          circle  square  triangle
circle        30       0         0
square         0      30         0
triangle       0       0        30
              precision    recall  f1-score   support

      circle     1.0000    1.0000    1.0000        30
      square     1.0000    1.0000    1.0000        30
    triangle     1.0000    1.0000    1.0000        30

    accuracy                         1.0000        90
   macro avg     1.0000    1.

## 정리 · README 정답 예와 대조

README 정답 실행 예: class_weights `[0.3000…, 0.9000…, 1.8000…]`, head/none 0.8111·0.8038·0.5, head/weight 1.0000·1.0000·1.0.

- **Accuracy는 높고 minority Recall은 0일 수 있는가?** 있다. 예를 들어 valid가 circle 90장·triangle 10장인데 모두 circle로 예측하면 Accuracy 0.90, triangle Recall 0이다. 이 실습의 valid는 균형(30·30·30)이라 head/none에서도 Accuracy 0.8111과 triangle Recall 0.5가 함께 나타났다.
- **weight와 sampler:** `CrossEntropyLoss(weight=...)`는 데이터는 그대로 두고 **손실에서 클래스별 비중**을 바꾼다. `WeightedRandomSampler`는 손실식은 그대로 두고 **배치에 뽑히는 빈도**를 바꾼다(소수 클래스를 더 자주 뽑음, 중복 추출). 이 실습은 weight만 실행했다.